<a href="https://colab.research.google.com/github/NikhilGeorge01/Cross-Lingual-Voice-Cloning/blob/main/VCT_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q streamlit pyngrok torch torchaudio soundfile openai-whisper deep-translator qwen-tts transformers librosa resemblyzer jiwer sacrebleu numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 9.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%%writefile app.py
import streamlit as st
import torch
import soundfile as sf
import whisper
from deep_translator import GoogleTranslator
from qwen_tts import Qwen3TTSModel  # ✅ Fixed import (TTS, not TT)
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification
import torchaudio
import numpy as np
import tempfile
import os
from pathlib import Path
from resemblyzer import VoiceEncoder, preprocess_wav
from sacrebleu.metrics import CHRF
from jiwer import wer

# ---------- Cached models ----------
@st.cache_resource
def load_whisper():
    return whisper.load_model("large")   # or "medium" if low RAM

@st.cache_resource
def load_tts():
    return Qwen3TTSModel.from_pretrained(  # ✅ Fixed class name
        "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
        device_map="cuda:0" if torch.cuda.is_available() else "cpu",
        dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    )

@st.cache_resource
def load_emotion(model_path="/content/drive/MyDrive/emotion_model"):
    processor = Wav2Vec2Processor.from_pretrained(model_path)
    model = Wav2Vec2ForSequenceClassification.from_pretrained(model_path)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    return processor, model, device

@st.cache_resource
def load_encoder():
    return VoiceEncoder()

# ---------- Emotion prediction ----------
def predict_emotion(audio_path, processor, model, device):
    wav, sr = torchaudio.load(audio_path)
    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)
    wav = wav.mean(dim=0)
    inputs = processor(wav.numpy(), sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        logits = model(inputs.input_values.to(device)).logits
    pred_id = logits.argmax(dim=1).item()
    return model.config.id2label[pred_id]

# ---------- Voice similarity ----------
def compute_similarity(ref_path, clone_path, encoder):
    ref_wav = preprocess_wav(Path(ref_path))
    clone_wav = preprocess_wav(Path(clone_path))
    ref_emb = encoder.embed_utterance(ref_wav)
    clone_emb = encoder.embed_utterance(clone_wav)
    return np.dot(ref_emb, clone_emb) / (np.linalg.norm(ref_emb) * np.linalg.norm(clone_emb))

# ---------- Streamlit UI ----------
st.set_page_config(page_title="Voice Cloning + Emotion", layout="wide")
st.title("🎙️ Cross-Lingual Voice Cloning with Emotion Analysis")
st.markdown("Clone your voice to **Telugu / Malayalam / Hindi** speech, then hear it in **English**.")

col1, col2 = st.columns(2)
with col1:
    st.subheader("1. Reference (English)")
    ref_audio = st.file_uploader("Upload your English reference (9-15 sec)", type=["wav","mp3","m4a"])
    ref_text = st.text_area("Exact text spoken in the reference", value="The quick brown fox jumps over the lazy dog near the riverbank every single morning.")
with col2:
    st.subheader("2. Source Audio (Foreign Language)")
    src_lang = st.selectbox("Select language", options=["te","ml","hi"], format_func=lambda x: {"te":"Telugu","ml":"Malayalam","hi":"Hindi"}[x])
    src_audio = st.file_uploader(f"Upload {src_lang} audio", type=["wav","mp3","m4a"])

if st.button("🚀 Translate & Clone & Detect Emotion"):
    if not ref_audio or not src_audio or not ref_text.strip():
        st.error("Please upload both files and provide reference text.")
        st.stop()

    # Save uploaded files to temp paths
    with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as f:
        f.write(ref_audio.read())
        ref_path = f.name
    with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as f:
        f.write(src_audio.read())
        src_path = f.name

    # Load models
    with st.spinner("Loading Whisper..."):
        whisper_model = load_whisper()
    with st.spinner("Loading Qwen3-TTS (first time downloads ~4.5GB)..."):
        tts_model = load_tts()
    with st.spinner("Loading emotion model..."):
        emot_processor, emot_model, emot_device = load_emotion()
    with st.spinner("Loading voice encoder..."):
        encoder = load_encoder()

    # Transcribe
    st.subheader("📝 Transcription & Translation")
    with st.spinner(f"Transcribing {src_lang}..."):
        result = whisper_model.transcribe(src_path, language=src_lang, task="transcribe", temperature=0.0, best_of=5, beam_size=5)
        source_text = result["text"].strip()
    st.write(f"**Original ({src_lang})**: {source_text}")

    # Translate
    with st.spinner("Translating to English..."):
        english_text = GoogleTranslator(source=src_lang, target="en").translate(source_text)
    st.write(f"**English translation**: {english_text}")

    # Voice clone
    st.subheader("🎤 Cloned Output")
    with st.spinner("Synthesizing in your cloned voice..."):
        wavs, sr = tts_model.generate_voice_clone(text=english_text, ref_audio=ref_path, ref_text=ref_text, language="English")
        with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as f:
            out_path = f.name
            sf.write(out_path, wavs[0], sr)
    st.audio(out_path, format="audio/wav")

    # Emotion
    st.subheader("😊 Emotion of Input Audio")
    emotion = predict_emotion(src_path, emot_processor, emot_model, emot_device)
    st.success(f"**Predicted emotion**: {emotion.upper()}")

    # Voice similarity
    st.subheader("🔊 Voice Similarity")
    cos_sim = compute_similarity(ref_path, out_path, encoder)
    st.metric("Cosine similarity", f"{cos_sim:.4f}")
    if cos_sim > 0.85:
        st.success("Excellent - very close")
    elif cos_sim > 0.70:
        st.info("Good - recognisable")
    elif cos_sim > 0.55:
        st.warning("Fair - some resemblance")
    else:
        st.error("Poor - voice drifted")

    # Optional translation quality
    st.subheader("📊 Translation Quality")
    back_translated = GoogleTranslator(source="en", target=src_lang).translate(english_text)
    st.write(f"**Back‑translated**: {back_translated}")
    chrf = CHRF()
    chrf_score = chrf.sentence_score(back_translated, [source_text])
    st.metric("ChrF score", f"{chrf_score.score:.1f}")
    try:
        wer_score = wer(source_text, back_translated)
        st.metric("Round‑trip WER", f"{wer_score:.3f}")
    except:
        pass

    # Cleanup
    os.unlink(ref_path); os.unlink(src_path); os.unlink(out_path)

st.caption("Built with Whisper, Qwen3‑TTS, and your trained emotion model.")

Writing app.py


In [4]:
!pip install -q pyngrok

from pyngrok import ngrok

# Kill any previous tunnels
ngrok.kill()

# Set your authtoken (sign up at ngrok.com and get it from https://dashboard.ngrok.com/auth)
# Replace "YOUR_AUTH_TOKEN" with your actual token
NGROK_AUTH_TOKEN = "YOUR_API_TOKEN"   # <---- CHANGE THIS
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Create a tunnel to port 8501 (default Streamlit port)
public_url = ngrok.connect(addr="8501", proto="http")
print(f"Streamlit app will be available at: {public_url}")

# Run Streamlit in the background
import subprocess
import time
process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"])
time.sleep(5)  # Give it a few seconds to start
print("Streamlit is running. Click the URL above to open the interface.")

Streamlit app will be available at: NgrokTunnel: "https://supremacy-nutlike-elk.ngrok-free.dev" -> "http://localhost:8501"
Streamlit is running. Click the URL above to open the interface.
